In [1]:
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import numpy as np


df = pd.read_csv('../data/raw/santander-customer-transaction-prediction/train.csv')
X = df.drop(["target", "ID_code"], axis=1)
y = df["target"]
test_df = pd.read_csv('../data/raw/santander-customer-transaction-prediction/test.csv')
X_test = test_df.drop("ID_code", axis=1)

In [2]:
from sklearn.model_selection import StratifiedKFold, train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [3]:
def run_cv(test_preds, min_child_samples, n_estimators, feature_fraction):
    oof_preds = np.zeros(len(X))
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_auc_scores = []
    best_iterations = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
        
        model = lgb.LGBMClassifier(
            n_estimators=n_estimators,
            learning_rate=0.05,
            random_state=42,
            num_leaves=16,
            min_child_samples=min_child_samples,
            feature_fraction=feature_fraction
        )
        
        model.fit(
            X_train_fold, y_train_fold,
            eval_set=[(X_val_fold, y_val_fold)],
            eval_metric="auc",
            callbacks=[
                lgb.early_stopping(stopping_rounds=100),
                lgb.log_evaluation(period=100)
            ]
        )
        
        fold_preds = model.predict_proba(X_val_fold)[:, 1]
        fold_auc = roc_auc_score(y_val_fold, fold_preds)
        fold_auc_scores.append(fold_auc)
        best_iterations.append(model.best_iteration_)

        oof_preds[val_idx] = fold_preds
        test_preds += (
            model.predict_proba(
                X_test,
                num_iteration=model.best_iteration_
            )[:, 1]
            / skf.n_splits
        )
        
        print(f"Fold {fold + 1} AUC: {fold_auc:.6f}")
        oof_auc = roc_auc_score(y, oof_preds)

    return {
        "test_preds": test_preds,
        "min_child_samples": min_child_samples,
        "n_estimators": n_estimators,
        "feature_fraction": feature_fraction,
        "mean_auc": np.mean(fold_auc_scores),
        "oof_auc": oof_auc,
        "std_auc": np.std(fold_auc_scores),
        "mean_best_iteration": np.mean(best_iterations),
        "best_iterations": best_iterations
    }

In [4]:
test_preds = np.zeros(len(X_test))
feature_fractions = [1.0, 0.8, 0.6, 0.4]
results = []
for feature_fraction in feature_fractions:
    res = run_cv(test_preds, min_child_samples=5000, n_estimators=5000, feature_fraction=feature_fraction)
    results.append(res)


[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Info] Number of positive: 16079, number of negative: 143921
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.058625 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 51000
[LightGBM] [Info] Number of data points in the train set: 160000, number of used features: 200
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.100494 -> initscore=-2.191750
[LightGBM] [Info] Start training from score -2.191750
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.841169	valid_0's binary_logloss: 0.267817
[20

In [5]:
for res in results:
    print(
        f"min_child_samples={res['min_child_samples']}, "
        f"feature_fraction={res['feature_fraction']}, "
        f"mean_auc={res['mean_auc']:.6f}, "
        f"std_auc={res['std_auc']:.6f}, "
        f"oof_auc={res['oof_auc']:.6f}, "
        f"mean_best_iteration={res['mean_best_iteration']:.6f}, "
        f"best_iterations={res['best_iterations']}"
    )

min_child_samples=5000, feature_fraction=1.0, mean_auc=0.896087, std_auc=0.002403, oof_auc=0.896075, mean_best_iteration=1470.000000, best_iterations=[1533, 1462, 1512, 1473, 1370]
min_child_samples=5000, feature_fraction=0.8, mean_auc=0.896182, std_auc=0.002484, oof_auc=0.896179, mean_best_iteration=1384.200000, best_iterations=[1501, 1356, 1354, 1403, 1307]
min_child_samples=5000, feature_fraction=0.6, mean_auc=0.896493, std_auc=0.002336, oof_auc=0.896408, mean_best_iteration=1510.200000, best_iterations=[1243, 1429, 1696, 1625, 1558]
min_child_samples=5000, feature_fraction=0.4, mean_auc=0.896738, std_auc=0.002803, oof_auc=0.896703, mean_best_iteration=1402.800000, best_iterations=[1411, 1323, 1480, 1437, 1363]


In [6]:
feature_fractions1 = [0.35, 0.3, 0.25, 0.2, 0.15, 0.1]
results1 = []
for feature_fraction in feature_fractions1:
    res = run_cv(test_preds, min_child_samples=5000, n_estimators=5000, feature_fraction=feature_fraction)
    results1.append(res)


[LightGBM] [Warning] feature_fraction is set=0.35, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.35
[LightGBM] [Warning] feature_fraction is set=0.35, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.35
[LightGBM] [Info] Number of positive: 16079, number of negative: 143921
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.047074 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 51000
[LightGBM] [Info] Number of data points in the train set: 160000, number of used features: 200
[LightGBM] [Warning] feature_fraction is set=0.35, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.100494 -> initscore=-2.191750
[LightGBM] [Info] Start training from score -2.191750
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.840752	valid_0's binary_logloss: 0.2688

In [7]:
for res in results1:
    print(
        f"min_child_samples={res['min_child_samples']}, "
        f"feature_fraction={res['feature_fraction']}, "
        f"mean_auc={res['mean_auc']:.6f}, "
        f"std_auc={res['std_auc']:.6f}, "
        f"oof_auc={res['oof_auc']:.6f}, "
        f"mean_best_iteration={res['mean_best_iteration']:.6f}, "
        f"best_iterations={res['best_iterations']}"
    )

min_child_samples=5000, feature_fraction=0.35, mean_auc=0.897105, std_auc=0.002575, oof_auc=0.897087, mean_best_iteration=1475.600000, best_iterations=[1417, 1443, 1523, 1490, 1505]
min_child_samples=5000, feature_fraction=0.3, mean_auc=0.896832, std_auc=0.002468, oof_auc=0.896732, mean_best_iteration=1376.400000, best_iterations=[1197, 1489, 1588, 1218, 1390]
min_child_samples=5000, feature_fraction=0.25, mean_auc=0.897290, std_auc=0.002215, oof_auc=0.897229, mean_best_iteration=1472.800000, best_iterations=[1290, 1510, 1596, 1576, 1392]
min_child_samples=5000, feature_fraction=0.2, mean_auc=0.897328, std_auc=0.001967, oof_auc=0.897209, mean_best_iteration=1541.000000, best_iterations=[1455, 1229, 1810, 1617, 1594]
min_child_samples=5000, feature_fraction=0.15, mean_auc=0.897478, std_auc=0.002422, oof_auc=0.897456, mean_best_iteration=1627.800000, best_iterations=[1563, 1638, 1721, 1646, 1571]
min_child_samples=5000, feature_fraction=0.1, mean_auc=0.897445, std_auc=0.002191, oof_auc=0

In [ ]:
feature_fractions2 = [0.18, 0.16, 0.14, 0.12, 0.08]
results2 = []
for feature_fraction in feature_fractions2:
    res = run_cv(test_preds, min_child_samples=5000, n_estimators=5000, feature_fraction=feature_fraction)
    results2.append(res)


[LightGBM] [Warning] feature_fraction is set=0.18, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.18


In [ ]:
for res in results2:
    print(
        f"min_child_samples={res['min_child_samples']}, "
        f"feature_fraction={res['feature_fraction']}, "
        f"mean_auc={res['mean_auc']:.6f}, "
        f"std_auc={res['std_auc']:.6f}, "
        f"oof_auc={res['oof_auc']:.6f}, "
        f"mean_best_iteration={res['mean_best_iteration']:.6f}, "
        f"best_iterations={res['best_iterations']}"
    )

In [ ]:
submission = pd.read_csv("../data/raw/santander-customer-transaction-prediction/sample_submission.csv")
submission["target"] = res["test_preds"]
submission.to_csv("../submissions/exp005.csv", index=False)